In [ ]:

import os

DATA_ROOT = "/kaggle/input"

for root, dirs, files in os.walk(DATA_ROOT):
    level = root.replace(DATA_ROOT, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}  {file}")

    # Avoid traversing excessively large directory trees
    dirs[:] = dirs[:10]


In [ ]:
import json
import numpy as np
from sklearn.model_selection import train_test_split

DATASET_PATH = "/kaggle/input/datasets/rhammell/ships-in-satellite-imagery/shipsnet.json"
SCENES_PATH = "/kaggle/input/datasets/rhammell/ships-in-satellite-imagery/scenes/scenes"

with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

raw = np.asarray(dataset["data"], dtype=np.float32)
y = np.asarray(dataset["labels"], dtype=np.int32)

X = raw.reshape(-1, 3, 80, 80).transpose(0, 2, 3, 1)

if X.max() <= 1:
    X *= 255

X = np.clip(X, 0, 255).astype(np.uint8)

print("Images:", X.shape)
print("Labels:", y.shape)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))
print("Pixel range:", X.min(), "to", X.max())
print("Scene files:", sorted(os.listdir(SCENES_PATH)))

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape, "Class counts:", np.bincount(y_train))
print("Validation:", X_val.shape, "Class counts:", np.bincount(y_val))
print("Test:", X_test.shape, "Class counts:", np.bincount(y_test))

In [ ]:
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

model = keras.Sequential([
    layers.Input(shape=(80, 80, 3)),
    layers.Rescaling(1.0 / 255),

    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.05),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.35),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc")
    ]
)

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weights = dict(zip(classes, weights))

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

model.save("/kaggle/working/ship_classifier_cnn.keras")
print("Model saved.")

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

test_probs = model.predict(X_test, batch_size=32, verbose=1).ravel()
test_preds = (test_probs >= 0.5).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_test, test_preds))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_preds,
    target_names=["No ship", "Ship"],
    digits=4
))

print("Test ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4))
print("Predicted ship patches:", int(test_preds.sum()), "/", len(test_preds))
print("Probability range:", float(test_probs.min()), "to", float(test_probs.max()))

In [ ]:
import os
import json

MODEL_PATH = "/kaggle/working/ship_classifier_cnn.keras"
METRICS_PATH = "/kaggle/working/patch_test_metrics.json"

assert os.path.exists(MODEL_PATH), "Model file was not found."

metrics = {
    "test_accuracy": 0.9350,
    "test_roc_auc": 0.9809,
    "ship_precision": 0.8581,
    "ship_recall": 0.8867,
    "ship_f1": 0.8721,
    "confusion_matrix": [[428, 22], [17, 133]]
}

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print("Model:", MODEL_PATH, os.path.getsize(MODEL_PATH), "bytes")
print("Metrics saved:", METRICS_PATH)

In [ ]:
from PIL import Image
import pandas as pd

WINDOW = 80
STRIDE = 40
THRESHOLD = 0.5491
NMS_IOU = 0.25
BATCH_SIZE = 128

def calculate_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - intersection
    return intersection / union if union else 0.0


def apply_nms(detections, iou_threshold=NMS_IOU):
    detections = sorted(detections, key=lambda d: d["score"], reverse=True)
    kept = []

    for detection in detections:
        if all(
            calculate_iou(detection["box"], previous["box"]) <= iou_threshold
            for previous in kept
        ):
            kept.append(detection)

    return kept


def scan_one_scene(image_path):
    image = Image.open(image_path).convert("RGB")
    image_array = np.asarray(image)
    height, width = image_array.shape[:2]

    xs = list(range(0, max(width - WINDOW + 1, 1), STRIDE))
    ys = list(range(0, max(height - WINDOW + 1, 1), STRIDE))

    if not xs or xs[-1] != width - WINDOW:
        xs.append(max(0, width - WINDOW))
    if not ys or ys[-1] != height - WINDOW:
        ys.append(max(0, height - WINDOW))

    positions = []
    patches = []

    for y0 in ys:
        for x0 in xs:
            patches.append(image_array[y0:y0 + WINDOW, x0:x0 + WINDOW])
            positions.append((x0, y0))

    probabilities = []

    for start in range(0, len(patches), BATCH_SIZE):
        batch = np.asarray(patches[start:start + BATCH_SIZE], dtype=np.uint8)
        probabilities.extend(
            model.predict(batch, verbose=0).ravel().tolist()
        )

    candidates = []

    for (x0, y0), score in zip(positions, probabilities):
        if score >= THRESHOLD:
            candidates.append({
                "box": [int(x0), int(y0), int(x0 + WINDOW), int(y0 + WINDOW)],
                "score": float(score)
            })

    final_detections = apply_nms(candidates)

    return {
        "image_size": [int(width), int(height)],
        "windows_scanned": len(patches),
        "candidate_count": len(candidates),
        "detections": final_detections
    }


print("Scanning functions are ready.")
print("Scenes found:", len([
    f for f in os.listdir(SCENES_PATH) if f.lower().endswith(".png")
]))

In [ ]:
import json
import time

scene_results = {}
summary_rows = []

scene_files = sorted(
    f for f in os.listdir(SCENES_PATH)
    if f.lower().endswith(".png")
)

for scene_name in scene_files:
    scene_path = os.path.join(SCENES_PATH, scene_name)

    start_time = time.time()
    result = scan_one_scene(scene_path)
    scene_results[scene_name] = result

    summary_rows.append({
        "scene": scene_name,
        "windows_scanned": result["windows_scanned"],
        "candidates": result["candidate_count"],
        "detections_after_nms": len(result["detections"]),
        "seconds": round(time.time() - start_time, 1)
    })

    # Save progress after every scene
    with open("/kaggle/working/scene_detections.json", "w") as f:
        json.dump(scene_results, f, indent=2)

    print(
        f"{scene_name}: "
        f"{result['windows_scanned']} windows, "
        f"{result['candidate_count']} candidates, "
        f"{len(result['detections'])} after NMS"
    )

scene_summary = pd.DataFrame(summary_rows)
scene_summary.to_csv(
    "/kaggle/working/scene_detection_summary.csv",
    index=False
)

print("\nSummary:")
display(scene_summary)

print("\nTotal windows:", scene_summary["windows_scanned"].sum())
print("Total candidates:", scene_summary["candidates"].sum())
print("Total detections after NMS:", scene_summary["detections_after_nms"].sum())
print("Saved scene detections to /kaggle/working/scene_detections.json")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

for scene_name, result in scene_results.items():
    image_path = os.path.join(SCENES_PATH, scene_name)

    with Image.open(image_path) as img:
        image = img.convert("RGB")
        fig, ax = plt.subplots(figsize=(14, 10))
        ax.imshow(image)

    for det in result["detections"]:
        x1, y1, x2, y2 = det["box"]
        score = det["score"]

        ax.add_patch(Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor="red", linewidth=1.5
        ))
        ax.text(
            x1, max(0, y1 - 5), f"{score:.2f}",
            color="yellow", fontsize=8,
            bbox=dict(facecolor="black", alpha=0.6, pad=1)
        )

    ax.set_title(
        f"{scene_name} — {len(result['detections'])} detections"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

visual_review = pd.DataFrame([
    {
        "scene": "lb_1.png",
        "boxes": 7,
        "false_positives_reported": 3,
        "missed_vessels_reported": None,
        "notes": "Remaining detections appear centered on ships."
    },
    {
        "scene": "lb_2.png",
        "boxes": 10,
        "false_positives_reported": None,
        "missed_vessels_reported": 5,
        "notes": "10 ships identified; 5 additional vessels reportedly missed."
    },
    {
        "scene": "lb_3.png",
        "boxes": 5,
        "false_positives_reported": None,
        "missed_vessels_reported": 4,
        "notes": "5 ships identified; 4 additional vessels reportedly missed."
    },
    {
        "scene": "lb_4.png",
        "boxes": 9,
        "false_positives_reported": 2,
        "missed_vessels_reported": None,
        "notes": "3 ships identified; remaining boxes need clarification."
    },
    {
        "scene": "sfbay_1.png",
        "boxes": 10,
        "false_positives_reported": 2,
        "missed_vessels_reported": 1,
        "notes": "8 boxes appear to contain ships."
    },
    {
        "scene": "sfbay_2.png",
        "boxes": 7,
        "false_positives_reported": 3,
        "missed_vessels_reported": 4,
        "notes": "4 boxes appear to contain ships."
    },
    {
        "scene": "sfbay_3.png",
        "boxes": 13,
        "false_positives_reported": 3,
        "missed_vessels_reported": None,
        "notes": "10 boxes appear to contain ships."
    },
    {
        "scene": "sfbay_4.png",
        "boxes": 1,
        "false_positives_reported": 1,
        "missed_vessels_reported": 1,
        "notes": "The detection appears incorrect."
    }
])

review_path = "/kaggle/working/scene_visual_review.csv"
visual_review.to_csv(review_path, index=False)

display(visual_review)
print("Saved:", review_path)

In [ ]:
TILE_SIZE = 400

tile_rows = []

for scene_name, result in scene_results.items():
    width, height = result["image_size"]

    # Count detections in each tile using each box's center
    tile_counts = {}

    for det in result["detections"]:
        x1, y1, x2, y2 = det["box"]
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        col = min(int(cx // TILE_SIZE), (width - 1) // TILE_SIZE)
        row = min(int(cy // TILE_SIZE), (height - 1) // TILE_SIZE)

        tile_counts[(row, col)] = tile_counts.get((row, col), 0) + 1

    n_rows = (height + TILE_SIZE - 1) // TILE_SIZE
    n_cols = (width + TILE_SIZE - 1) // TILE_SIZE

    for row in range(n_rows):
        for col in range(n_cols):
            x_start = col * TILE_SIZE
            y_start = row * TILE_SIZE
            x_end = min(x_start + TILE_SIZE, width)
            y_end = min(y_start + TILE_SIZE, height)

            tile_rows.append({
                "scene": scene_name,
                "row": row,
                "col": col,
                "x1": x_start,
                "y1": y_start,
                "x2": x_end,
                "y2": y_end,
                "area": (x_end - x_start) * (y_end - y_start),
                "detection_count": tile_counts.get((row, col), 0)
            })

tiles_df = pd.DataFrame(tile_rows)

print("Scenes:", tiles_df["scene"].nunique())
print("Total tiles:", len(tiles_df))
print("\nDetection-count distribution:")
display(
    tiles_df["detection_count"]
    .value_counts()
    .sort_index()
    .rename_axis("detections_per_tile")
    .to_frame("number_of_tiles")
)

print("\nOverall summary:")
display(tiles_df["detection_count"].describe())

In [ ]:
from scipy.stats import nbinom
from statsmodels.stats.multitest import multipletests

ALPHA_PRIOR = 0.5
BETA_PRIOR = 2.0  # Gamma rate parameter
FULL_TILE_AREA = TILE_SIZE * TILE_SIZE

anomaly_rows = []

for scene_name in tiles_df["scene"].unique():
    scene_tiles = tiles_df[tiles_df["scene"] == scene_name].copy()

    # Leave-one-tile-out baseline within this scene
    scene_total_count = scene_tiles["detection_count"].sum()
    scene_total_area = scene_tiles["area"].sum()

    for idx, tile in scene_tiles.iterrows():
        other_count = scene_total_count - tile["detection_count"]
        other_area = scene_total_area - tile["area"]

        # Posterior Gamma parameters for rate per pixel
        alpha_post = ALPHA_PRIOR + other_count
        beta_post = BETA_PRIOR + other_area / FULL_TILE_AREA

        # Exposure expressed in full-tile units
        exposure = tile["area"] / FULL_TILE_AREA

        # Negative-binomial posterior predictive parameters
        p = beta_post / (beta_post + exposure)
        observed_count = int(tile["detection_count"])

        tail_probability = nbinom.sf(
            observed_count - 1,
            alpha_post,
            p
        )

        predictive_mean = alpha_post * exposure / beta_post

        anomaly_rows.append({
            "scene": scene_name,
            "row": int(tile["row"]),
            "col": int(tile["col"]),
            "detection_count": observed_count,
            "tile_area": int(tile["area"]),
            "baseline_mean": predictive_mean,
            "tail_p_value": tail_probability
        })

anomaly_df = pd.DataFrame(anomaly_rows)

# Correct across all 240 tested tiles
anomaly_df["q_value"] = multipletests(
    anomaly_df["tail_p_value"],
    method="fdr_bh"
)[1]

anomaly_df = anomaly_df.sort_values(
    ["q_value", "tail_p_value"]
).reset_index(drop=True)

print("Tiles analyzed:", len(anomaly_df))
print("Tiles with q < 0.05:", int((anomaly_df["q_value"] < 0.05).sum()))

print("\nTop 15 tiles by raw tail probability:")
display(
    anomaly_df[
        [
            "scene", "row", "col", "detection_count",
            "baseline_mean", "tail_p_value", "q_value"
        ]
    ].head(15).round(5)
)

In [ ]:
# Inspect the most statistically significant tile
target_scene = "lb_3.png"
target_row = 3
target_col = 1

tile = tiles_df[
    (tiles_df["scene"] == target_scene) &
    (tiles_df["row"] == target_row) &
    (tiles_df["col"] == target_col)
].iloc[0]

scene_path = os.path.join(SCENES_PATH, target_scene)
scene_image = Image.open(scene_path).convert("RGB")

x1, y1, x2, y2 = int(tile["x1"]), int(tile["y1"]), int(tile["x2"]), int(tile["y2"])
tile_image = scene_image.crop((x1, y1, x2, y2))

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(tile_image)
ax.set_title(
    f"{target_scene} | tile ({target_row}, {target_col})\n"
    f"Detections: {int(tile['detection_count'])} | "
    f"Bayesian predictive mean: {anomaly_df[(anomaly_df['scene'] == target_scene) & (anomaly_df['row'] == target_row) & (anomaly_df['col'] == target_col)]['baseline_mean'].iloc[0]:.3f}"
)
ax.axis("off")
plt.show()

# Show detector boxes that overlap this tile
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(scene_image)

for det in scene_results[target_scene]["detections"]:
    bx1, by1, bx2, by2 = det["box"]
    cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2

    if x1 <= cx < x2 and y1 <= cy < y2:
        ax.add_patch(Rectangle(
            (bx1, by1), bx2 - bx1, by2 - by1,
            fill=False, edgecolor="red", linewidth=2
        ))
        ax.text(
            bx1, max(0, by1 - 5), f"{det['score']:.2f}",
            color="yellow", fontsize=10,
            bbox=dict(facecolor="black", alpha=0.7, pad=2)
        )

ax.set_xlim(x1, x2)
ax.set_ylim(y2, y1)
ax.set_title(f"Detections in significant tile: {target_scene}, row {target_row}, col {target_col}")
ax.axis("off")
plt.show()

In [ ]:
# Compare simple count threshold with Bayesian anomaly flags

comparison_df = tiles_df.merge(
    anomaly_df[
        ["scene", "row", "col", "tail_p_value", "q_value", "baseline_mean"]
    ],
    on=["scene", "row", "col"],
    how="left"
)

comparison_df["baseline_flag"] = (
    comparison_df["detection_count"] >= 2
)

comparison_df["bayesian_flag"] = (
    comparison_df["q_value"] < 0.05
)

print("Tiles evaluated:", len(comparison_df))
print("Simple baseline flags:", int(comparison_df["baseline_flag"].sum()))
print("Bayesian flags:", int(comparison_df["bayesian_flag"].sum()))

print("\nBayesian-flagged tiles:")
display(
    comparison_df[comparison_df["bayesian_flag"]][
        ["scene", "row", "col", "detection_count",
         "baseline_mean", "tail_p_value", "q_value"]
    ].round(5)
)

print("\nTiles flagged by the simple baseline but not Bayesian:")
display(
    comparison_df[
        comparison_df["baseline_flag"] &
        ~comparison_df["bayesian_flag"]
    ][
        ["scene", "row", "col", "detection_count",
         "baseline_mean", "tail_p_value", "q_value"]
    ].sort_values("q_value").head(15).round(5)
)

In [ ]:
# Display the 9 tiles flagged by the simple baseline but not Bayesian
rejected_tiles = comparison_df[
    comparison_df["baseline_flag"] &
    ~comparison_df["bayesian_flag"]
].copy()

n = len(rejected_tiles)
fig, axes = plt.subplots(
    (n + 2) // 3, 3,
    figsize=(15, 5 * ((n + 2) // 3))
)
axes = np.array(axes).reshape(-1)

for ax, (_, tile) in zip(axes, rejected_tiles.iterrows()):
    scene_name = tile["scene"]
    scene_image = Image.open(
        os.path.join(SCENES_PATH, scene_name)
    ).convert("RGB")

    x1, y1, x2, y2 = map(
        int, [tile["x1"], tile["y1"], tile["x2"], tile["y2"]]
    )
    ax.imshow(scene_image.crop((x1, y1, x2, y2)))
    ax.set_title(
        f"{scene_name} | tile ({int(tile['row'])}, {int(tile['col'])})\n"
        f"Count={int(tile['detection_count'])}, "
        f"mean={tile['baseline_mean']:.3f}, "
        f"q={tile['q_value']:.3f}"
    )
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Manual validation of the nine tiles rejected by Bayesian scoring.
# Values are based on your visual inspection.

manual_labels = [
    {"tile_number": 1, "valid_vessels": 2, "detector_count": 2, "notes": "Both detections correct"},
    {"tile_number": 2, "valid_vessels": 1, "detector_count": 3, "notes": "One vessel"},
    {"tile_number": 3, "valid_vessels": 1, "detector_count": 2, "notes": "One likely vessel"},
    {"tile_number": 4, "valid_vessels": 2, "detector_count": 3, "notes": "Two actual ships"},
    {"tile_number": 5, "valid_vessels": 2, "detector_count": 2, "notes": "Both correct"},
    {"tile_number": 6, "valid_vessels": 1, "detector_count": 2, "notes": "One vessel"},
    {"tile_number": 7, "valid_vessels": 0, "detector_count": 2, "notes": "No vessel"},
    {"tile_number": 8, "valid_vessels": 2, "detector_count": 2, "notes": "Both correct"},
    {"tile_number": 9, "valid_vessels": 1, "detector_count": 2, "notes": "One vessel; second partly visible"}
]

manual_df = pd.DataFrame(manual_labels)
manual_df["estimated_false_positives"] = (
    manual_df["detector_count"] - manual_df["valid_vessels"]
)

display(manual_df)

print("Tiles reviewed:", len(manual_df))
print("Detector outputs:", manual_df["detector_count"].sum())
print("Estimated valid detections:", manual_df["valid_vessels"].sum())
print("Estimated false positives:", manual_df["estimated_false_positives"].sum())

In [ ]:
# Prepare all non-empty tiles for manual validation

review_df = comparison_df[
    comparison_df["detection_count"] > 0
].copy()

review_df = review_df.sort_values(
    ["detection_count", "scene", "row", "col"],
    ascending=[False, True, True, True]
).reset_index(drop=True)

review_df["manual_valid_vessels"] = np.nan
review_df["manual_false_positives"] = np.nan
review_df["review_status"] = "not reviewed"

print("Non-empty tiles to review:", len(review_df))
print("Total detector outputs in these tiles:",
      int(review_df["detection_count"].sum()))

display(
    review_df[
        ["scene", "row", "col", "detection_count",
         "baseline_mean", "q_value", "review_status"]
    ].head(20).round(4)
)

In [ ]:
# Visual review batch 1: first 10 non-empty tiles

batch = review_df.head(10)

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
axes = axes.flatten()

for ax, (_, tile) in zip(axes, batch.iterrows()):
    scene_name = tile["scene"]
    scene_image = Image.open(
        os.path.join(SCENES_PATH, scene_name)
    ).convert("RGB")

    x1, y1, x2, y2 = map(
        int, [tile["x1"], tile["y1"], tile["x2"], tile["y2"]]
    )

    ax.imshow(scene_image.crop((x1, y1, x2, y2)))
    ax.set_title(
        f"Image {int(tile.name)}: {scene_name}\n"
        f"Tile ({int(tile['row'])}, {int(tile['col'])}) | "
        f"Count: {int(tile['detection_count'])}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Summarize existing detection confidence scores without rerunning the model

all_scores = []

for scene_name, result in scene_results.items():
    for det in result["detections"]:
        all_scores.append({
            "scene": scene_name,
            "confidence": float(det["score"])
        })

scores_df = pd.DataFrame(all_scores)

print("Total detections:", len(scores_df))
print("\nConfidence summary:")
display(scores_df["confidence"].describe().round(4))

print("\nConfidence distribution:")
display(
    scores_df.assign(
        confidence_band=pd.cut(
            scores_df["confidence"],
            bins=[0, 0.60, 0.80, 0.90, 1.0],
            include_lowest=True,
            labels=["≤0.60", "0.60–0.80", "0.80–0.90", "0.90–1.00"]
        )
    )["confidence_band"].value_counts().sort_index().rename("detections")
)

In [ ]:
# Compare raw detection counts, confidence-weighted scores, and Bayesian results.
# Reuses existing scene_results, tiles_df, and anomaly_df.

weighted_rows = []

for scene_name, result in scene_results.items():
    width, height = result["image_size"]

    for det in result["detections"]:
        bx1, by1, bx2, by2 = det["box"]
        cx = (bx1 + bx2) / 2
        cy = (by1 + by2) / 2

        col = min(int(cx // TILE_SIZE), (width - 1) // TILE_SIZE)
        row = min(int(cy // TILE_SIZE), (height - 1) // TILE_SIZE)

        weighted_rows.append({
            "scene": scene_name,
            "row": row,
            "col": col,
            "confidence": float(det["score"])
        })

det_scores_df = pd.DataFrame(weighted_rows)

weighted_summary = (
    det_scores_df.groupby(["scene", "row", "col"])
    .agg(
        score_sum=("confidence", "sum"),
        score_mean=("confidence", "mean"),
        high_conf_count=("confidence", lambda s: int((s >= 0.90).sum()))
    )
    .reset_index()
)

comparison_df = (
    comparison_df.drop(
        columns=["score_sum", "score_mean", "high_conf_count"],
        errors="ignore"
    )
    .merge(
        weighted_summary,
        on=["scene", "row", "col"],
        how="left"
    )
)

comparison_df[["score_sum", "score_mean", "high_conf_count"]] = (
    comparison_df[["score_sum", "score_mean", "high_conf_count"]].fillna(0)
)

print("Tiles analyzed:", len(comparison_df))
print("Detections assigned to tiles:", int(comparison_df["detection_count"].sum()))

print("\nTop 10 tiles by raw detection count:")
display(
    comparison_df.sort_values(
        ["detection_count", "scene"],
        ascending=[False, True]
    )[
        ["scene", "row", "col", "detection_count",
         "score_sum", "score_mean", "high_conf_count", "q_value"]
    ].head(10).round(4)
)

print("\nTop 10 tiles by confidence-weighted score:")
display(
    comparison_df.sort_values(
        ["score_sum", "scene"],
        ascending=[False, True]
    )[
        ["scene", "row", "col", "detection_count",
         "score_sum", "score_mean", "high_conf_count", "q_value"]
    ].head(10).round(4)
)

In [ ]:
# Sensitivity analysis across CNN confidence thresholds.
# No model retraining or manual review required.

thresholds = [0.55, 0.60, 0.70, 0.80, 0.90, 0.95]

sensitivity_rows = []

for threshold in thresholds:
    retained = det_scores_df[
        det_scores_df["confidence"] >= threshold
    ]

    tile_counts = (
        retained.groupby(["scene", "row", "col"])
        .size()
        .rename("retained_count")
        .reset_index()
    )

    # Include all tiles, including tiles with zero retained detections.
    result = tiles_df[
        ["scene", "row", "col"]
    ].drop_duplicates().merge(
        tile_counts,
        on=["scene", "row", "col"],
        how="left"
    )

    result["retained_count"] = (
        result["retained_count"].fillna(0).astype(int)
    )

    sensitivity_rows.append({
        "confidence_threshold": threshold,
        "detections_retained": len(retained),
        "tiles_with_detections": int(
            (result["retained_count"] > 0).sum()
        ),
        "tiles_with_2plus": int(
            (result["retained_count"] >= 2).sum()
        ),
        "tiles_with_3plus": int(
            (result["retained_count"] >= 3).sum()
        ),
        "max_detections_in_tile": int(
            result["retained_count"].max()
        )
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)

display(sensitivity_df)

In [ ]:
print("tiles_df columns:")
print(tiles_df.columns.tolist())

print("\nanomaly_df columns:")
print(anomaly_df.columns.tolist())

In [ ]:

from scipy.stats import nbinom
from statsmodels.stats.multitest import multipletests

# Refit the Bayesian anomaly analysis at each confidence threshold.
# Uses existing det_scores_df and tiles_df.
# Prior: Gamma(shape=0.5, rate=2.0).

ALPHA_PRIOR = 0.5
BETA_PRIOR = 2.0
thresholds = [0.55, 0.60, 0.70, 0.80, 0.90, 0.95]

bayesian_sensitivity = []
bayesian_sensitivity_details = {}

for threshold in thresholds:
    retained = det_scores_df[
        det_scores_df["confidence"] >= threshold
    ]

    counts = (
        retained.groupby(["scene", "row", "col"])
        .size()
        .rename("detection_count")
        .reset_index()
    )

    current = (
        tiles_df.drop(columns=["detection_count"], errors="ignore")
        .merge(counts, on=["scene", "row", "col"], how="left")
    )
    current["detection_count"] = (
        current["detection_count"].fillna(0).astype(int)
    )

    # Leave-one-tile-out Gamma-Poisson predictive scoring,
    # calculated separately within each scene.
    current["baseline_mean"] = 0.0
    current["tail_p_value"] = 1.0

    for scene_name, indices in current.groupby("scene").groups.items():
        scene = current.loc[indices]
        observed = scene["detection_count"].to_numpy()
        areas = scene["area"].to_numpy()

        # Estimate the background rate from the other tiles
        # in the same scene, weighted by tile exposure/area.
        for idx in indices:
            mask = scene.index != idx
            other_counts = scene.loc[mask, "detection_count"].to_numpy()
            other_areas = scene.loc[mask, "area"].to_numpy()

            total_count = other_counts.sum()
            total_area = other_areas.sum()

            rate = (
                (ALPHA_PRIOR + total_count)
                / (BETA_PRIOR + total_area / 400**2)
            )

            exposure = current.loc[idx, "area"] / 400**2
            predictive_mean = rate * exposure

            # Gamma-Poisson predictive distribution = Negative Binomial.
            shape = ALPHA_PRIOR + total_count
            prob = (
                (BETA_PRIOR + total_area / 400**2)
                / (BETA_PRIOR + total_area / 400**2 + exposure)
            )

            k = int(current.loc[idx, "detection_count"])
            p_value = nbinom.sf(k - 1, shape, prob) if k > 0 else 1.0

            current.loc[idx, "baseline_mean"] = predictive_mean
            current.loc[idx, "tail_p_value"] = p_value

    current["q_value"] = multipletests(
        current["tail_p_value"].to_numpy(),
        method="fdr_bh"
    )[1]

    current["confidence_threshold"] = threshold
    bayesian_sensitivity_details[threshold] = current.copy()

    bayesian_sensitivity.append({
        "confidence_threshold": threshold,
        "detections_retained": int(current["detection_count"].sum()),
        "tiles_tested": len(current),
        "tiles_flagged_q_lt_0.05": int((current["q_value"] < 0.05).sum()),
        "tiles_flagged_q_lt_0.10": int((current["q_value"] < 0.10).sum()),
        "minimum_q_value": float(current["q_value"].min()),
    })

bayesian_sensitivity_df = pd.DataFrame(bayesian_sensitivity)
display(bayesian_sensitivity_df.round(5))

print("\nTiles flagged at q < 0.05 for each threshold:")
for threshold, current in bayesian_sensitivity_details.items():
    flagged = current[current["q_value"] < 0.05]
    print(f"\nThreshold {threshold:.2f}: {len(flagged)} flagged")
    if len(flagged):
        display(
            flagged[
                ["scene", "row", "col", "detection_count",
                 "baseline_mean", "tail_p_value", "q_value"]
            ].sort_values("q_value").round(5)
        )
